# Simulate DDF

- from Madeleine Ginolin
- adapted by SDC
- **date** : `2026-10-26`
- **kernel on my laptop** : conda_py313
- **kernel on NERSC** : desc-td-env (Not -dev)
  
The adaptation wrt the original is on the downloading of DUST maps

## Overview of the notebook (explanatory comments added on 2026-10-29)

This notebook simulates **Type Ia supernovae (SNe Ia) in the six LSST Deep Drilling Fields (DDFs)** with `skysurvey`, using the real per-CCD observing log of the DP2/EDP2 DDF visits, and then tests how well their light curves can be fitted with SALT3 (`sncosmo`).

Workflow:

1. **Setup**: detect the machine (NERSC or laptop), configure `dustmaps` and download the SFD dust maps (needed for the Milky Way extinction `mwebv`).
2. **Population model**: SALT3 template, asymmetric-Gaussian distributions of stretch `x1` and colour `c`, Tripp standardisation, Frohmaier+2019 rate, F99 Milky Way dust.
3. **Survey**: read the per-CCD visit table, define a square CCD footprint, register the Rubin bandpasses in `sncosmo`.
4. **Simulation**: for each DDF, draw SNe Ia (0.05 < z < 1.2) and generate their noisy observed light curves.
5. **Processing**: put fluxes on a common zero point (ZP = 30), co-add per night, apply the SNANA search efficiency, then require 5-sigma detections in at least 3 bands.
6. **SALT3 fits**: fit each SN (a) with free redshift and (b) with the redshift fixed to its true value, apply fit-quality cuts and count the SNe per field.
7. **Validation**: pull distributions (fit - truth) / error for x1, c, t0, m_B and x0, and an example light curve with the true and fitted models.

Note: the selection / counting cells that follow each of the two fits (57-61 and 64-68) are the same code; the variables are overwritten, so the pulls and plots at the end describe the second fit (redshift fixed).

In [ ]:
# Imports: skysurvey (simulation of transients observed by a survey), sncosmo (SN models, bandpasses, light-curve fitting),
# astropy / scipy / numpy / pandas (cosmology, units, interpolation, statistics, tables), shapely (CCD footprint geometry),
# matplotlib (plots), socket (host name, used below to detect NERSC vs laptop).
import skysurvey
import numpy as np
import os
import pandas
import sncosmo
from scipy.stats import norm, lognorm
import scipy
import astropy
from astropy import cosmology
from scipy.special import expit
import time
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from shapely import geometry
from astropy.coordinates import SkyCoord
import astropy.units as u
import socket

In [ ]:
# Environment check: print the Python interpreter / version and where the key packages are imported from,
# to make sure the right kernel is used (conda_py313 on the laptop, desc-td-env on NERSC).
import sys
import importlib.util

print("Python executable:", sys.executable)
print("Python version:", sys.version)

for package in ["skysurvey", "sncosmo", "astropy", "numpy", "dustmaps"]:
    spec = importlib.util.find_spec(package)
    print(f"{package:12s}", spec.origin if spec else "NOT INSTALLED")

## Where I work

In [ ]:
# Choose the directory where the dust maps are stored, depending on the machine:
#   - NERSC (host name contains 'nersc' or starts with perlmutter / nid / login): ./ASTROMAPS in the working directory
#   - otherwise (laptop): ~/DATA/ASTROMAPS
# The directory is registered in the dustmaps configuration so that dustmaps looks for its maps there.
from dustmaps.config import config

hostname = socket.gethostname().lower()

if "nersc" in hostname or hostname.startswith(("perlmutter", "nid", "login")):
    # NERSC
    cwd = os.getcwd()
    data_dir = os.path.join(os.getcwd(), "ASTROMAPS")  # Replace with your actual NERSC path
    os.makedirs(data_dir, exist_ok=True)
    print("Directory created:", data_dir)
else:
    # Local Mac
    data_dir = os.path.join(os.path.expanduser("~"), "DATA", "ASTROMAPS")

config["data_dir"] = data_dir
sfd_dir = os.path.join(data_dir, "sfd")

print("Hostname:", hostname)
print("dustmaps data_dir:", config["data_dir"])
print("SFD maps directory:", os.path.join(data_dir, "sfd"))

## Need Dust maps

In [ ]:
# Check that dustmaps (with its SFD module) and requests are installed, and print their versions / locations.
import os
import dustmaps
import dustmaps.sfd
import requests
from importlib.metadata import version, PackageNotFoundError

try:
    print("dustmaps version:", version("dustmaps"))
except PackageNotFoundError:
    print("Métadonnées du paquet dustmaps introuvables")

print("dustmaps module:", dustmaps.__file__)
print("SFD module:", dustmaps.sfd.__file__)
print("Requests version:", requests.__version__)

### The right way to  import Dustmaps

- import on Mylaptop
- According ChatGPT

In [ ]:
# Download the two SFD dust-map FITS files (north and south Galactic hemispheres) from the NERSC portal into the sfd
# directory, unless they are already there. The data are streamed in 1 MB chunks to a temporary '.part' file that is
# renamed only when the download is complete, so an interrupted download never leaves a truncated map behind.
import os
import requests

# path_home = os.path.expanduser("~")
# data_dir = os.path.join(path_home, "DATA", "ASTROMAPS", "sfd")
# os.makedirs(data_dir, exist_ok=True)
os.makedirs(sfd_dir, exist_ok=True)

base_url = "https://portal.nersc.gov/project/cosmo/data/dust/v0_1/maps/"

filenames = [
    "SFD_dust_4096_ngp.fits",
    "SFD_dust_4096_sgp.fits",
]

for filename in filenames:
    # destination = os.path.join(data_dir, filename)
    destination = os.path.join(sfd_dir, filename)

    # Skip download if the file already exists and is not empty
    if os.path.isfile(destination) and os.path.getsize(destination) > 0:
        print(f"Already present, skipping: {destination}")
        continue

    url = base_url + filename
    print(f"Downloading {filename}...")

    try:
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()

        # Write to a temporary file to avoid leaving a partial download
        temp_destination = destination + ".part"

        with open(temp_destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

        if os.path.getsize(temp_destination) == 0:
            raise IOError("Downloaded file is empty")

        os.replace(temp_destination, destination)
        print(f"Saved: {destination}")

    except Exception as e:
        print(f"Download failed for {filename}: {e}")

        if os.path.exists(temp_destination):
            os.remove(temp_destination)

### Check the DUST maps

In [ ]:
# Integrity check of the downloaded maps: print the shape, dtype and size on disk of each FITS file.
from astropy.io import fits

for filename in filenames:
    filepath = os.path.join(sfd_dir, filename)

    with fits.open(filepath) as hdul:
        print(filename)
        print("Dimensions :", hdul[0].data.shape)
        print("Type :", hdul[0].data.dtype)
        print("Taille :", os.path.getsize(filepath) / 1e6, "MB")

## Use DUST maps

In [ ]:
# Print the data directory currently used by dustmaps (the commented lines show how to set it by hand).
# from dustmaps.config import config
# path_home = os.path.expanduser("~")
# config["data_dir"] = os.path.join(path_home, "DATA", "ASTROMAPS")
print("dustmaps data directory:", config["data_dir"])

In [ ]:
# Instantiate the SFD query object (E(B-V) lookup on the SFD maps); this also checks that the maps can be read from the
# configured directory. The Milky Way extinction of the simulated SNe (get_mwebv below) relies on the same dustmaps setup.
from dustmaps.sfd import SFDQuery

sfd = SFDQuery(map_dir=os.path.join(config["data_dir"], "sfd"))

# Simulations

### Define the SN Ia model

In [ ]:
# Two helper functions used by the SN Ia population model defined in the next cell.
# Asymmetric gaussian for x1 and c to match SNANA (the sigma_right/left parametrisation is also inherited from there)
def asymmetric_gaussian(xx, mu, sigma_left, sigma_right):
    if type(xx) is str:  # assumed r_ input
        xx = eval(f"np.r_[{xx}]")
    # sigma = np.where(xx < mu, sigma_left, sigma_right)
    # Width = sigma_left below the mean mu and sigma_right above it
    sigma = np.ones(len(xx)) * sigma_right
    sigma[xx < mu] = sigma_left
    # Normalisation so that the density integrates to 1; the function returns the grid and the density evaluated on it
    norm = np.sqrt(2 / np.pi) / (sigma_left + sigma_right)
    return xx, norm * np.exp(-0.5 * ((xx - mu) / sigma) ** 2)


# Frohmaier rate (updated to match the last skysurvey update)
# Volumetric SN Ia rate as a function of redshift: r0 * (1 + z)^alpha (r0 in events per Gpc^3 per year)
def frohmaier_2019(z, r0=2.3e4, alpha=1.70):
    return r0 * (1 + z) ** alpha

In [ ]:
# SN Ia population model, written as a dictionary of parameters (a small graph in skysurvey):
# each entry says how a parameter is drawn, and '@name' refers to another parameter of the graph.
# SALT3 template (spectral energy distribution) used to generate the light curves:
sncosmo_model = sncosmo.Model(source=sncosmo.get_source("salt3"))

# Distributions of the intrinsic and geometric parameters of each SN
SNeIa = dict(
    # Redshift, drawn according to the volumetric rate (zmax here is superseded by the zmin / zmax given to snia.draw below)
    redshift={"kwargs": {"zmax": 0.2}, "as": "z"},
    # SALT stretch x1 and colour c: asymmetric Gaussians with SNANA parameters (grid given as 'start:stop:step')
    x1={
        "func": asymmetric_gaussian,
        "kwargs": {"xx": "-4:4:0.005", "mu": 0.973, "sigma_left": 1.472, "sigma_right": 0.222},
    },
    c={
        "func": asymmetric_gaussian,
        "kwargs": {"xx": "-0.3:1:0.001", "mu": -0.054, "sigma_left": 0.043, "sigma_right": 0.101},
    },
    # Date of peak brightness (MJD), uniform; this range is superseded by tstart / tstop given to snia.draw below
    t0={"func": np.random.uniform, "kwargs": {"low": 56_000, "high": 56_200}},
    # Absolute magnitude from the Tripp (1998) relation: mabs + alpha * x1 + beta * c, with Gaussian intrinsic scatter sigmaint
    magabs={
        "func": skysurvey.target.snia.SNeIaMagnitude.tripp1998,  # the @ points towards another parameter in the graph
        "kwargs": {"x1": "@x1", "c": "@c", "mabs": -19.3, "sigmaint": 0.1, "alpha": -0.15, "beta": 3.15},
    },
    # Apparent magnitude = absolute magnitude + distance modulus at redshift z
    magobs={
        "func": "magabs_to_magobs",  # str-> method of the class
        "kwargs": {"z": "@z", "magabs": "@magabs"},
    },
    # Random sky position; superseded by the ra / dec ranges given to snia.draw for each field
    radec={"func": skysurvey.tools.utils.random_radec, "kwargs": {}, "as": ["ra", "dec"]},
)

# F99 dust to match the dust law used in DES analysis (I think a paper found that it matches the data better than F99)
dust_effect = {
    "effect": sncosmo.models.F99Dust(r_v=3.1),
    "name": "mw",
    "frame": "obs",
    # Milky Way reddening E(B-V) at the SN position, read from the SFD dust map
    "model": {
        "mwebv": {
            "func": skysurvey.effects.milkyway.get_mwebv,
            "kwargs": {"ra": "@ra", "dec": "@dec", "which": "sfd"},
        }
    },
}

### Load the survey

In [ ]:
# Load the observing log: one row per (visit, detector) for the DDF visits (EDP2 detector-visit table, from Bastien).
# Columns used below: field, band, mjd, ra, dec; the others (e.g. zero point) are passed on to skysurvey.
# Now using per CCD logs (from Bastien's file)
logs_full = pandas.read_parquet("edp2_detectorvisit_table.parquet")
# Manually calculated the area of the footprint of a given CCD
footprint_ccd = 9.6 / 189
# Area of one CCD = 9.6 deg^2 (LSSTCam field of view) / 189 (science CCDs); its square root is the side of the square footprint (deg)
side_ccd = np.sqrt(footprint_ccd)
# Define a square footprint per CCD
footprint = geometry.Polygon(
    (
        (-side_ccd / 2, -side_ccd / 2),
        (-side_ccd / 2, side_ccd / 2),
        (side_ccd / 2, side_ccd / 2),
        (side_ccd / 2, -side_ccd / 2),
    )
)

In [ ]:
# Register the Rubin bandpasses in sncosmo. The transmission curves (wavelength in nm, transmission) come from the butler
# and are stored in filter_lsst_<band>.npy; they are registered as 'rubinu' ... 'rubiny' (trimmed and normalised), and the
# 'band' column of the observing log is renamed lsstX -> rubinX so that the simulation uses these bandpasses.
# Using bands from the butler (will update once/if there is an official sourcefile)
bands = ["u", "g", "r", "i", "z", "y"]
for band in bands:
    filename = "filter_lsst_" + band + ".npy"
    bandpass = np.load(filename)
    # Here we trim the light curves to prevent a bug, but if/where we cut is TBD with the whole WG
    sncosmo.register(
        sncosmo.Bandpass(
            bandpass[0], bandpass[1], wave_unit=u.nm, name="rubin" + band, trim_level=0.001, normalize=True
        ),
        force=True,
    )

    logs_full.loc[logs_full["band"] == "lsst" + band, "band"] = "rubin" + band

Note: Bug (when floating $z$) if all bands span 300 - 1100 nm, as sncosmo requires that the SN model is defined throughout the whole wavelength of a band to compute fluxes.

In [ ]:
# Display the observing log (columns and number of rows).
logs_full

In [ ]:
# Plot the Rubin bandpasses registered above (solid) against sncosmo's built-in LSST bandpasses (grey dashed) to compare them.
# (The registration itself was done in the previous cell; the curves have no labels, so plt.legend() has nothing to show.)
# Register the bands in sncosmo
bands = ["u", "g", "r", "i", "z", "y"]
for band in bands:
    bandpass = sncosmo.get_bandpass("rubin" + band)
    plt.plot(bandpass.wave, bandpass.trans)
    bandpass_sncosmo = sncosmo.get_bandpass("lsst" + band)
    plt.plot(bandpass_sncosmo.wave, bandpass_sncosmo.trans, c="grey", ls="dashed")
plt.legend()
plt.show()

### Setup the skysurvey object and simulate

In [ ]:
# Build the skysurvey SN Ia target and attach the population model, the SALT3 template, the cosmology,
# the volumetric rate and the Milky Way dust effect defined above. Nothing is drawn yet.
snia = skysurvey.SNeIa()
snia.set_model(SNeIa)
snia.set_template(sncosmo_model)
# Cosmo agreed at the DESC July 2026 meeting (should double check that we are using actually the same)
snia.set_cosmology(cosmology.FlatLambdaCDM(H0=70, Om0=0.315))
snia.set_rate(frohmaier_2019)
snia.add_effect(dust_effect)

In [ ]:
%%time

# MAIN SIMULATION LOOP, done one Deep Drilling Field at a time (COSMOS, ECDFS, EDFS_a, EDFS_b, ELAISS1, XMM_LSS).
# For each field: build the survey from its visits, draw SNe Ia, then 'observe' them to get noisy light curves.
ddf_fields = ["COSMOS", "ECDFS", "EDFS_a", "EDFS_b", "ELAISS1", "XMM_LSS"]

# Containers: true SN parameters and simulated light-curve points, one DataFrame per field
snias_full = []
lcs_full = []

# Here we simulate field per field - no longer needed with the fix of the skyarea bug - TO UPDATE

for field in ddf_fields:
    # Visits / CCDs of this field, and the corresponding skysurvey Survey (pointings + square CCD footprint)
    logs = logs_full[logs_full["field"] == field]
    lsst = skysurvey.Survey.from_pointings(logs, footprint=footprint)
    # Draw the true SNe of the field: 0.05 < z < 1.2, peak dates within the survey duration +/- 30 days, positions in the
    # bounding box of the pointings (+ one CCD side); set_amplitude=True also computes the flux amplitude x0 from the magnitude
    field_snias = snia.draw(
        zmin=0.05,
        zmax=1.2,
        tstart=min(lsst.data["mjd"]) - 30,
        tstop=max(lsst.data["mjd"]) + 30,
        set_amplitude=True,
        radec={
            "ra_range": [min(logs["ra"]) - side_ccd, max(logs["ra"]) + side_ccd],
            "dec_range": [min(logs["dec"]) - side_ccd, max(logs["dec"]) + side_ccd],
        },
    )
    # Unique name for each SN, and record of its field
    field_snias["sn"] = ["LSST_" + field + "_" + str(k) for k in list(field_snias.index)]
    field_snias["field"] = field
    snia.set_data(field_snias)

    snias_full.append(field_snias)

    # Observe the SNe with the survey: noisy light curves (flux, fluxerr, zp, band, mjd) for phases between -20 and +40 days
    # around peak; only SNe that fall on a CCD at an observed date give points
    dset = skysurvey.DataSet.from_targets_and_survey(snia, lsst, incl_error=True, phase_range=[-20, 40])
    index = list(dset.data.index.get_level_values(0))
    # Attach the SN name (first index level = SN index) and the magnitude system (AB)
    field_lcs = dset.data.assign(sn=["LSST_" + field + "_" + str(ind) for ind in index], magsys="ab")

    lcs_full.append(field_lcs)

In [ ]:
# Merge the per-field results: true SN parameters (snia_data) and all simulated light-curve points (lcs).
snia_data = pandas.concat(snias_full)
lcs = pandas.concat(lcs_full)

In [ ]:
# Columns of the true-parameter table (e.g. z, t0, x1, c, x0, mwebv, ra, dec, sn, field).
snia_data.keys()

In [ ]:
lcs

In [ ]:
%%time
# Express all fluxes and errors at a common zero point ZP = 30: flux' = flux * 10^(0.4 * (30 - zp)),
# so that points taken with different zero points can be combined.
# Put the LCs on the same ZP=30
lcs["flux_prime"] = lcs["flux"] * 10 ** (0.4 * (30 - lcs["zp"]))
lcs["fluxerr_prime"] = lcs["fluxerr"] * 10 ** (0.4 * (30 - lcs["zp"]))

In [ ]:
# Mask over the true-SN table: True for SNe that have at least one simulated observation.
mask_observed = snia_data["sn"].isin(np.unique(lcs["sn"]))

In [ ]:
# Names of the six DDFs (same list as in the simulation loop).
fields = ["COSMOS", "ECDFS", "EDFS_a", "EDFS_b", "ELAISS1", "XMM_LSS"]

In [ ]:
# Number of SNe with at least one observation, per field.
for field in fields:
    print(field + ": " + str(len(snia_data[(snia_data["field"] == field) & mask_observed])) + " SNe")

In [ ]:
lcs

Note: here is where we should trim the LCs based on available templates. For example XMM_LSS does not have any co-adds, hence no templates, so that there will be no SNe in this DDF in the actual sample

### LC coadds

In [ ]:
# Co-add the light curves per night, SN and band, with an inverse-variance weighted mean:
#   flux = sum(w * flux_i) / sum(w),  fluxerr = 1 / sqrt(sum(w)),  with w = 1 / fluxerr_i^2
# Coadding the LCs per night

lcs["mjd_round"] = lcs["mjd"].copy()  # Rounding the MJD to get the night (only works for Chile/US/Hawaii)
lcs["mjd_round"] = lcs["mjd_round"].map(int)
lcs["weights"] = 1 / (lcs["fluxerr_prime"] ** 2)
lcs["flux_weighted"] = lcs["flux_prime"] * lcs["weights"]

# Sums of w * flux and of w over each (SN, band, night) group
flux_sum = lcs[["band", "flux_weighted", "mjd_round", "sn"]].copy().groupby(["sn", "band", "mjd_round"]).sum()
weight_sum = lcs[["band", "weights", "mjd_round", "sn"]].copy().groupby(["sn", "band", "mjd_round"]).sum()

lcs_coadd = flux_sum.join(weight_sum)

lcs_coadd["flux"] = lcs_coadd["flux_weighted"] / lcs_coadd["weights"]  # Weighted mean of the fluxes
lcs_coadd["fluxerr"] = 1 / np.sqrt(lcs_coadd["weights"])
lcs_coadd.drop(columns=["flux_weighted", "weights"], inplace=True)
# Move band and night back to columns (the index keeps the SN name)
lcs_coadd.reset_index(level=[1, 2], inplace=True)
# Signal-to-noise ratio of each co-added point (called ndetection): used for the search efficiency and the detection cuts
lcs_coadd["ndetection"] = lcs_coadd["flux"] / lcs_coadd["fluxerr"]

In [ ]:
# Co-added fluxes are at ZP = 30 in the AB system (columns required by sncosmo).
lcs_coadd["zp"] = 30
lcs_coadd["zpsys"] = "ab"

### Searcheff

In [ ]:
# Search efficiency: probability that a point of a given signal-to-noise ratio (SNR) passes the photometry / detection pipeline.
# The tabulated curves come from SNANA (estimated from Sanchez+22 on DC2 Run2.2i WFD); there is one curve per band,
# interpolated with a cubic spline as a function of SNR.
# Search eff (probability of a point of a given SNR to go through the photometry pipeline, taken from SNANA)

# bands = ['lsstu', 'lsstg', 'lsstr', 'lssti', 'lsstz', 'lssty']
bands = ["rubinu", "rubing", "rubinr", "rubini", "rubinz", "rubiny"]
filepath = "SEARCHEFF_PIPELINE_LSST_DC2_Run2.2i_WFD.DAT"  # Estimated from Sanchez+22
# SNR grid of the table (99 values, second column of the file)
sigma = np.loadtxt(filepath, skiprows=27, max_rows=99, usecols=[1])

searcheff_funcs = []  # Defining the search_eff functions
for i in range(len(bands)):
    # Efficiency curve of band i: the file holds the 6 bands one after the other, in blocks of 101 lines
    searcheff_list = np.loadtxt(filepath, skiprows=27 + i * 101, max_rows=99, usecols=[2])
    searcheff_funcs.append(CubicSpline(sigma, searcheff_list))

In [ ]:
# Apply the search efficiency to the co-added points: evaluate the efficiency of the band at each point's SNR
# (clipped to [0, 1], NaN -> 0), keep each point with that probability (binomial draw) and drop the others.
searcheff = np.zeros(len(lcs_coadd))  # Calculating the probability of a point to pass through the pipeline
for i in range(len(bands)):
    mask_bands = lcs_coadd["band"] == bands[i]
    func = searcheff_funcs[i]
    searcheff[mask_bands] = func(lcs_coadd[mask_bands]["ndetection"])
searcheff[searcheff < 0] = 0
searcheff[searcheff > 1] = 1
searcheff[np.isnan(searcheff)] = 0

lcs_coadd["search_eff"] = searcheff
lcs_coadd["selected"] = np.random.binomial(1, lcs_coadd["search_eff"])  # Drawing from a binomial law
lcs_coadd = lcs_coadd[lcs_coadd["selected"] == 1]  # Only keeping the selected points
lcs_coadd.drop(columns=["selected", "search_eff"], inplace=True)
lcs_coadd.reset_index(inplace=True)

### LC quality cuts

In [ ]:
# Work on copies, so that the full simulated tables (lcs_coadd, snia_data) are kept intact by the selection steps below.
lcs_select = lcs_coadd.copy()
snias = snia_data.copy()
snias_new = snias.copy()

In [ ]:
# Light-curve quality cuts, SN by SN and band by band, on the co-added points:
#   - a point is a detection if flux / fluxerr >= nsigma_det
#   - rest-frame phase = (MJD - t0) / (1 + z)
#   - ndet counts detections with -10 < phase < 40 days; ndet_pre: -10 < phase < 0; ndet_post: 0 < phase < 40
#   - nbands = number of bands with at least one detection; ndetection* = total numbers of detections per SN
# Cuts on the LC: how many n sigma detections, in how many bands, how many pre- and post-peak detections

nsigma_det = 5  # Level of detection considered

lcs_select["ndetection"] = lcs_select["flux"] / lcs_select["fluxerr"]
lcs_select.set_index("sn", inplace=True)
snias.set_index("sn", inplace=True)
# Attach the true t0, z, x1 and c of each SN to its light-curve points (merge on the SN name)
data_merged = lcs_select.merge(snias[["t0", "z", "x1", "c"]], on="sn")
data_merged["phase_obs"] = data_merged["mjd_round"] - data_merged["t0"]
data_merged["phase"] = data_merged["phase_obs"] / (1 + data_merged["z"])
ndet = (
    data_merged[(data_merged["ndetection"] >= nsigma_det) & (data_merged["phase"].between(-10, +40))]
    .groupby(["sn", "band"])
    .size()
)
ndet_pre = (
    data_merged[(data_merged["ndetection"] >= nsigma_det) & (data_merged["phase"].between(-10, 0))]
    .groupby(["sn", "band"])
    .size()
)
ndet_post = (
    data_merged[(data_merged["ndetection"] >= nsigma_det) & (data_merged["phase"].between(0, +40))]
    .groupby(["sn", "band"])
    .size()
)
nbands = ndet.groupby(level=0).size()
ndetection = ndet.groupby(level=0).sum()
ndetection_pre = ndet_pre.groupby(level=0).sum()
ndetection_post = ndet_post.groupby(level=0).sum()

Note: Here only 5$\sigma$ detections in three bands, but we could play around with that depending on what is done in the data (see Rebecca's DP2 slides)

In [ ]:
flag_used = (
    nbands >= 3
)  # & (ndetection_pre>=1) & (ndetection_post>=1) #& (ndetection>7) & (ndetection_pre>=2) & (ndetection_post>=2)
targets_to_consider = ndetection[flag_used].index

In [ ]:
snias_detected = snia_data[snia_data["sn"].isin(targets_to_consider)]

In [ ]:
for field in fields:
    print(field + ": " + str(len(snias_detected[snias_detected["field"] == field])) + " SNe")

- No template in XMM_LSS

In [ ]:
lcs_coadd.rename(columns={"mjd_round": "time", "magsys": "zpsys"}, inplace=True)

In [ ]:
lcs_coadd[lcs_coadd["sn"] == targets_to_consider[0]]

# SALT fits

- With or woithout Redshift
- No redshift from host gaalxy
- Perhaps host Galaxy

In [ ]:
def fit_salt_integrated(lc, z_guess, t0_guess, mwebv_guess, fit_z=False):
    model = sncosmo.Model(
        source=sncosmo.get_source("salt3"),
        effects=[sncosmo.F99Dust(r_v=3.1)],
        effect_names=["mw"],
        effect_frames=["obs"],
    )
    keymap = {}
    lc_dict = {
        key: lc[keymap.get(key, key)].values for key in ["time", "band", "flux", "fluxerr", "zp", "zpsys"]
    }
    if fit_z == False:
        model.set(z=z_guess, t0=t0_guess, mwebv=mwebv_guess)
        try:
            result, fitted_model = sncosmo.fit_lc(
                lc_dict,
                model,
                ["t0", "x0", "x1", "c"],  # parameters of model to vary
                bounds={"x1": (-10, 10), "c": (-1, 3)},
                modelcov=True,
            )  # bounds on parameters (if any)
        except RuntimeError:
            return np.ones(24) * np.nan
        if result["success"] == False:
            return np.ones(24) * np.nan
        return np.concatenate(
            [
                result["parameters"],
                result["covariance"].flatten(),
                np.array([result["chisq"], result["ndof"]]),
            ]
        )
    else:
        model.set(t0=t0_guess, mwebv=mwebv_guess)
        try:
            result, fitted_model = sncosmo.fit_lc(
                lc_dict,
                model,
                ["t0", "x0", "x1", "c", "z"],  # parameters of model to vary
                bounds={"x1": (-10, 10), "c": (-1, 3), "z": (0, 0.5)},
                modelcov=True,
            )  # bounds on parameters (if any)
        except RuntimeError:
            return np.ones(33) * np.nan
        if result["success"] == False:
            return np.ones(33) * np.nan
        return np.concatenate(
            [
                result["parameters"],
                result["covariance"].flatten(),
                np.array([result["chisq"], result["ndof"]]),
            ]
        )

### SALT fits without z info

In [ ]:
%%time

salt_fits = []

fit_z = True

for i in range(len(targets_to_consider)):
    sn_name = targets_to_consider[i]
    lc = lcs_coadd[lcs_coadd["sn"] == targets_to_consider[i]]
    sn = snias_new[snias_new["sn"] == targets_to_consider[i]]
    df_result = fit_salt_integrated(
        lc,
        z_guess=sn["z"].iloc[0],
        t0_guess=sn["t0"].iloc[0] + norm.rvs(0, 10),
        mwebv_guess=sn["mwebv"].iloc[0],
        fit_z=fit_z,
    )
    salt_fits.append(np.array(df_result))
salt_fits = np.array(salt_fits)

if fit_z == False:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
else:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_z_z",
            "cov_z_t0",
            "cov_z_x0",
            "cov_z_x1",
            "cov_z_c",
            "cov_t0_z",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_z",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_z",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_z",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
    df_salt["z_err"] = np.sqrt(df_salt["cov_z_z"])
df_salt["t0_err"] = np.sqrt(df_salt["cov_t0_t0"])
df_salt["x0_err"] = np.sqrt(df_salt["cov_x0_x0"])
df_salt["x1_err"] = np.sqrt(df_salt["cov_x1_x1"])
df_salt["c_err"] = np.sqrt(df_salt["cov_c_c"])
df_salt["sn"] = list(targets_to_consider)
df_salt["fitprob"] = scipy.stats.chi2.sf(df_salt["chisq"], df_salt["ndof"])

In [ ]:
mask_c = df_salt["c"].between(-0.3, 0.3) & (df_salt["c_err"] < 0.2)
mask_x1 = df_salt["x1"].between(-3, 3) & (df_salt["x1_err"] < 1)
mask_fit = (df_salt["t0_err"] < 2) & (df_salt["fitprob"] > 1e-4) & (df_salt["mwebv"] < 0.3)

df_salt_selected = df_salt[mask_c & mask_x1 & mask_fit]
snias_new_selected = snias_new[snias_new["sn"].isin(df_salt_selected["sn"])]

In [ ]:
df_salt_selected.sort_values(by="sn", inplace=True)
df_salt_selected.reset_index(inplace=True)
snias_new_selected.sort_values(by="sn", inplace=True)
snias_new_selected.reset_index(inplace=True)

In [ ]:
df_salt_selected["mb"] = -2.5 * np.log10(df_salt_selected["x0"]) + 10.502
df_salt_selected["mb_err"] = 2.5 * df_salt_selected["x0_err"] / (np.log(10) * df_salt_selected["x0"])
snias_new_selected["mb"] = -2.5 * np.log10(snias_new_selected["x0"]) + 10.502

In [ ]:
print("Sucessful SALT fit")
for field in fields:
    print(
        field
        + ": "
        + str(
            len(
                snias_detected[
                    (snias_detected["field"] == field) & (snias_detected["sn"].isin(df_salt.dropna()["sn"]))
                ]
            )
        )
        + " SNe"
    )

In [ ]:
print("After SALT cuts")
for field in fields:
    print(field + ": " + str(len(snias_new_selected[snias_new_selected["field"] == field])) + " SNe")

### SALT fit with z info

In [ ]:
%%time

salt_fits = []

fit_z = False

for i in range(len(targets_to_consider)):
    sn_name = targets_to_consider[i]
    lc = lcs_coadd[lcs_coadd["sn"] == targets_to_consider[i]]
    sn = snias_new[snias_new["sn"] == targets_to_consider[i]]
    df_result = fit_salt_integrated(
        lc,
        z_guess=sn["z"].iloc[0],
        t0_guess=sn["t0"].iloc[0] + norm.rvs(0, 10),
        mwebv_guess=sn["mwebv"].iloc[0],
        fit_z=fit_z,
    )
    salt_fits.append(np.array(df_result))
salt_fits = np.array(salt_fits)

if fit_z == False:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
else:
    df_salt = pandas.DataFrame(
        salt_fits,
        columns=[
            "z",
            "t0",
            "x0",
            "x1",
            "c",
            "mwebv",
            "cov_z_z",
            "cov_z_t0",
            "cov_z_x0",
            "cov_z_x1",
            "cov_z_c",
            "cov_t0_z",
            "cov_t0_t0",
            "cov_t0_x0",
            "cov_t0_x1",
            "cov_t0_c",
            "cov_x0_z",
            "cov_x0_t0",
            "cov_x0_x0",
            "cov_x0_x1",
            "cov_x0_c",
            "cov_x1_z",
            "cov_x1_t0",
            "cov_x1_x0",
            "cov_x1_x1",
            "cov_x1_c",
            "cov_c_z",
            "cov_c_t0",
            "cov_c_x0",
            "cov_c_x1",
            "cov_c_c",
            "chisq",
            "ndof",
        ],
    )
    df_salt["z_err"] = np.sqrt(df_salt["cov_z_z"])
df_salt["t0_err"] = np.sqrt(df_salt["cov_t0_t0"])
df_salt["x0_err"] = np.sqrt(df_salt["cov_x0_x0"])
df_salt["x1_err"] = np.sqrt(df_salt["cov_x1_x1"])
df_salt["c_err"] = np.sqrt(df_salt["cov_c_c"])
df_salt["sn"] = list(targets_to_consider)
df_salt["fitprob"] = scipy.stats.chi2.sf(df_salt["chisq"], df_salt["ndof"])

In [ ]:
mask_c = df_salt["c"].between(-0.3, 0.3) & (df_salt["c_err"] < 0.2)
mask_x1 = df_salt["x1"].between(-3, 3) & (df_salt["x1_err"] < 1)
mask_fit = (df_salt["t0_err"] < 2) & (df_salt["fitprob"] > 1e-4) & (df_salt["mwebv"] < 0.3)

df_salt_selected = df_salt[mask_c & mask_x1 & mask_fit]
snias_new_selected = snias_new[snias_new["sn"].isin(df_salt_selected["sn"])]

In [ ]:
df_salt_selected["mb"] = -2.5 * np.log10(df_salt_selected["x0"]) + 10.502
df_salt_selected["mb_err"] = 2.5 * df_salt_selected["x0_err"] / (np.log(10) * df_salt_selected["x0"])
snias_new_selected["mb"] = -2.5 * np.log10(snias_new_selected["x0"]) + 10.502

In [ ]:
df_salt_selected.sort_values(by="sn", inplace=True)
df_salt_selected.reset_index(inplace=True)
snias_new_selected.sort_values(by="sn", inplace=True)
snias_new_selected.reset_index(inplace=True)

In [ ]:
print("Sucessful SALT fit")
for field in fields:
    print(
        field
        + ": "
        + str(
            len(
                snias_detected[
                    (snias_detected["field"] == field) & (snias_detected["sn"].isin(df_salt.dropna()["sn"]))
                ]
            )
        )
        + " SNe"
    )

In [ ]:
print("After SALT cuts")
for field in fields:
    print(field + ": " + str(len(snias_new_selected[snias_new_selected["field"] == field])) + " SNe")

In [ ]:
pull_x1 = (df_salt_selected["x1"] - snias_new_selected["x1"]) / df_salt_selected["x1_err"]
pull_c = (df_salt_selected["c"] - snias_new_selected["c"]) / df_salt_selected["c_err"]
pull_t0 = (df_salt_selected["t0"] - snias_new_selected["t0"]) / df_salt_selected["t0_err"]
pull_mb = (df_salt_selected["mb"] - snias_new_selected["mb"]) / df_salt_selected["mb_err"]
pull_x0 = (df_salt_selected["x0"] - snias_new_selected["x0"]) / df_salt_selected["x0_err"]

In [ ]:
nbins = 30

bins_pull = np.linspace(-5, 5, nbins)
xpull = np.linspace(-5, 5, 200)

fig, ax = plt.subplots(4, 3, figsize=(14, 8))

ax[0, 0].hist(snias_new_selected["x1"], bins=np.linspace(-3, 3, nbins), color="teal", alpha=0.3)
ax[0, 0].set_xlabel(r"True $x_1$")
ax[1, 0].hist(pull_x1, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 0].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 0].set_xlabel(r"Pull on $x_1$")
ax[1, 0].legend()
ax[0, 0].set_ylabel(r"Number of SNe")
ax[1, 0].set_ylabel(r"PDF")


ax[0, 1].hist(snias_new_selected["c"], bins=np.linspace(-0.3, 0.8, nbins), color="teal", alpha=0.3)
ax[0, 1].set_xlabel(r"True $c$")
ax[1, 1].hist(pull_c, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 1].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 1].set_xlabel(r"Pull on $c$")

ax[0, 2].hist(df_salt_selected["t0"], bins=np.linspace(60823, 61060, nbins), color="teal", alpha=0.3)
ax[0, 2].set_xlabel(r"True $t_0$")
ax[1, 2].hist(pull_t0, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[1, 2].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[1, 2].set_xlabel(r"Pull on $t_0$")

ax[2, 0].hist(df_salt_selected["mb"], bins=np.linspace(19, 25, nbins), color="teal", alpha=0.3)
ax[2, 0].set_xlabel(r"$m_B$")
ax[3, 0].hist(pull_mb, bins=bins_pull, color="darkred", alpha=0.5, density=True)
ax[3, 0].plot(xpull, norm.pdf(xpull), color="Black", alpha=0.5, label=r"$\mathcal{N(0,1)}$")
ax[3, 0].set_xlabel(r"Pull on $m_B$")
ax[2, 0].set_ylabel(r"Number of SNe")
ax[3, 0].set_ylabel(r"PDF")

ax[2, 1].hist(df_salt_selected["z"], bins=np.linspace(0, 1, nbins), color="teal", alpha=0.3)
ax[2, 1].set_xlabel(r"$z$")

for i in range(4):
    for j in range(3):
        ax[i, j].spines["top"].set_visible(False)
        ax[i, j].spines["right"].set_visible(False)
        ax[i, j].spines["left"].set_visible(False)
        ax[i, j].spines["bottom"].set_visible(False)

fig.suptitle("All DDFs, " + str(int(len(pull_x1))) + " SNe using true $z$")

fig.delaxes(ax[3, 2])
fig.delaxes(ax[2, 2])
# fig.delaxes(ax[2,1])
fig.delaxes(ax[3, 1])


plt.tight_layout()
# plt.savefig('/Users/mginolin/Documents/stats_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')

plt.show()

### Plot a LC

In [ ]:
sn_name = snias_new_selected.sample(n=1)["sn"].iloc[0]
# sn_name = 'LSST_63'
sn = snias_new_selected[snias_new_selected["sn"] == sn_name]
lc = lcs_coadd[lcs_coadd["sn"] == sn_name]
lc_bis = lcs[lcs["sn"] == sn_name]
print(sn["sn"].iloc[0])

In [ ]:
sn_fit = df_salt_selected[df_salt_selected["sn"] == sn["sn"].iloc[0]]

In [ ]:
dust = sncosmo.F99Dust(r_v=3.1)
true_model = sncosmo.Model(source="salt3", effects=[dust], effect_names=["mw"], effect_frames=["obs"])
# Don't forget to add the MW extinction to the SALT model
params = {
    "z": sn["z"].iloc[0],
    "t0": sn["t0"].iloc[0],
    "x0": sn["x0"].iloc[0],
    "x1": sn["x1"].iloc[0],
    "c": sn["c"].iloc[0],
    "mwebv": sn["mwebv"].iloc[0],
}
true_model.update(params)
phase_min = sn["t0"].iloc[0] - 40
phase_max = sn["t0"].iloc[0] + 60

In [ ]:
salt_fit = sncosmo.Model(source="salt3", effects=[dust], effect_names=["mw"], effect_frames=["obs"])
params = {
    "z": sn_fit["z"].iloc[0],
    "t0": sn_fit["t0"].iloc[0],
    "x0": sn_fit["x0"].iloc[0],
    "x1": sn_fit["x1"].iloc[0],
    "c": sn_fit["c"].iloc[0],
    "mwebv": sn["mwebv"].iloc[0],
}
salt_fit.update(params)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# Converting the phase from rest- to observer-frame
xphase = np.linspace(
    sn["t0"].iloc[0] - 40 * (1 + sn["z"].iloc[0]), sn["t0"].iloc[0] + 60 * (1 + sn["z"].iloc[0]), 1000
)

peaktime = sn["t0"].iloc[0]

bands = np.unique(np.unique(lc["band"]))
c = ["C0", "C1", "C2", "C3", "C4", "C5"]

for i in range(len(bands)):
    band = bands[i]
    lc_band = lc[(lc["band"] == band) & (lc["time"].between(phase_min, phase_max))]
    lc_band_bis = lc_bis[(lc_bis["band"] == band) & (lc_bis["mjd"].between(phase_min, phase_max))]
    ax.scatter(lc_band["time"] - peaktime, lc_band["flux"], s=5, label=band, c=c[i])
    ax.errorbar(
        lc_band["time"] - peaktime,
        lc_band["flux"],
        yerr=lc_band["fluxerr"],
        fmt="none",
        zorder=2,
        alpha=0.5,
        c=c[i],
    )
    ax.plot(
        xphase - peaktime, true_model.bandflux(band, xphase, zp=30, zpsys="ab"), ls="dashed", c=c[i], lw=1
    )
    ax.plot(xphase - peaktime, salt_fit.bandflux(band, xphase, zp=30, zpsys="ab"), c=c[i], lw=1)

ax.plot([-100, -90], [0, 0], lw=1, c="Black", ls="dashed", label="True SALT3 model")
ax.plot([-100, -90], [0, 0], lw=1, c="Black", label="Fitted SALT3 model")

# ax.text(-40, 375, r'$z^\mathrm{true}=$'+str(round(sn['z'].iloc[0], 3))+r', $x_1^\mathrm{true}=$'+str(round(sn['x1'].iloc[0], 3))+r', $c^\mathrm{true}=$'+str(round(sn['c'].iloc[0], 3)))
# ax.text(-40, 400, r'$z^\mathrm{fit}=$'+str(round(sn_fit['z'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['z_err'].iloc[0], 3))+r', $x_1^\mathrm{fit}=$'+str(round(sn_fit['x1'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['x1_err'].iloc[0], 3))+r', $c^\mathrm{fit}=$'+str(round(sn_fit['c'].iloc[0], 3))+r'$\pm$'+str(round(sn_fit['c_err'].iloc[0], 3)))
# Annotations are placed in axes coordinates (0-1) so they always stay inside the figure
ax.text(
    0.02,
    0.97,
    r"$z^\mathrm{true}=$"
    + str(round(sn["z"].iloc[0], 3))
    + r", $x_1^\mathrm{true}=$"
    + str(round(sn["x1"].iloc[0], 3))
    + r", $c^\mathrm{true}=$"
    + str(round(sn["c"].iloc[0], 3)),
    transform=ax.transAxes,
    va="top",
)
ax.text(
    0.02,
    0.90,
    r"$x_1^\mathrm{fit}=$"
    + str(round(sn_fit["x1"].iloc[0], 3))
    + r"$\pm$"
    + str(round(sn_fit["x1_err"].iloc[0], 3))
    + r", $c^\mathrm{fit}=$"
    + str(round(sn_fit["c"].iloc[0], 3))
    + r"$\pm$"
    + str(round(sn_fit["c_err"].iloc[0], 3)),
    transform=ax.transAxes,
    va="top",
)

# Leave headroom above the data so the annotations do not overlap the curves
ax.set_ylim(top=ax.get_ylim()[1] * 1.35)
ax.legend(loc="upper right", ncol=2, frameon=False)

ax.set_yticks([0])
ax.set_ylabel("Flux")
ax.set_xlabel(r"Observer-frame phase (in days)")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.spines["bottom"].set_visible(False)

ax.set_xlim(-40, 60)
fig.tight_layout()
# plt.savefig('/Users/mginolin/Documents/example_LC_elaiss_DP2.pdf', format='pdf', dpi=1200, bbox_inches='tight')
plt.show()